score = 0.561

In [14]:
import os
from pathlib import Path
import datetime
from typing import List
from tqdm import tqdm
from dataclasses import dataclass, asdict

import polars as pl 
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.preprocessing import StandardScaler

import kaggle_evaluation.default_inference_server

In [15]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [16]:
# ============ PATHS ============
from pathlib import Path

KAGGLE_DATA_PATH: Path = Path('/kaggle/input/hull-tactical-market-prediction/')
LOCAL_DATA_PATH: Path = Path('./data')

# Prefer Kaggle input path when available, otherwise fall back to a local ./data folder.
DATA_PATH: Path = KAGGLE_DATA_PATH if KAGGLE_DATA_PATH.exists() else LOCAL_DATA_PATH

# ============ RETURNS TO SIGNAL CONFIGS ============
MIN_SIGNAL: float = 0.0                         # Minimum value for the daily signal 
MAX_SIGNAL: float = 2.0                         # Maximum value for the daily signal 
SIGNAL_MULTIPLIER: float = 400.0                # Multiplier of predicted excess returns to signal 

# ============ MODEL CONFIGS (RIDGE) ============
CV: int = 10                                    # Number of cross validation folds
ALPHAS: np.ndarray = np.logspace(-4, 2, 100)    # Candidate ridge alphas


In [17]:
@dataclass
class DatasetOutput:
    X_train : pl.DataFrame 
    X_test: pl.DataFrame
    y_train: pl.Series
    y_test: pl.Series
    scaler: StandardScaler

@dataclass 
class RidgeParameters:
    cv: int
    alphas: np.ndarray

    def __post_init__(self):
        if self.cv < 2:
            raise ValueError("cv must be >= 2 for cross-validation")
        if self.alphas is None or len(self.alphas) == 0:
            raise ValueError("alphas must be a non-empty array")

@dataclass(frozen=True)
class RetToSignalParameters:
    signal_multiplier: float 
    min_signal : float = MIN_SIGNAL
    max_signal : float = MAX_SIGNAL


In [18]:
ret_signal_params = RetToSignalParameters(
    signal_multiplier= SIGNAL_MULTIPLIER
)

ridge_params = RidgeParameters(
    cv = CV,
    alphas = ALPHAS
)


In [19]:

def load_trainset() -> pl.DataFrame:
    """
    Loads and preprocesses the training dataset.

    Returns:
        pl.DataFrame: The preprocessed training DataFrame.
    """
    return (
        pl.read_csv(DATA_PATH / "train.csv")
        .rename({'market_forward_excess_returns':'target'})
        .with_columns(
            pl.exclude('date_id').cast(pl.Float64, strict=False)
        )
        .head(-10)
    )

def load_testset() -> pl.DataFrame:
    """
    Loads and preprocesses the testing dataset.

    Returns:
        pl.DataFrame: The preprocessed testing DataFrame.
    """
    return (
        pl.read_csv(DATA_PATH / "test.csv")
        .rename({'lagged_forward_returns':'target'})
        .with_columns(
            pl.exclude('date_id').cast(pl.Float64, strict=False)
        )
    )

def create_example_dataset(df: pl.DataFrame) -> pl.DataFrame:
    """
    Creates new features and cleans a DataFrame.

    Args:
        df (pl.DataFrame): The input Polars DataFrame.

    Returns:
        pl.DataFrame: The DataFrame with new features, selected columns, and no null values.
    """
    vars_to_keep: List[str] = [
        "S2", "E2", "E3", "P9", "S1", "S5", "I2", "P8",
        "P10", "P12", "P13", "U1", "U2"
    ]

    return (
        df.with_columns(
            (pl.col("I2") - pl.col("I1")).alias("U1"),
            (pl.col("M11") / ((pl.col("I2") + pl.col("I9") + pl.col("I7")) / 3)).alias("U2")
        )
        .select(["date_id", "target"] + vars_to_keep)
        .with_columns([
            pl.col(col).fill_null(pl.col(col).ewm_mean(com=0.5))
            for col in vars_to_keep
        ])
        .drop_nulls()
    )
    
def join_train_test_dataframes(train: pl.DataFrame, test: pl.DataFrame) -> pl.DataFrame:
    """
    Joins two dataframes by common columns and concatenates them vertically.

    Args:
        train (pl.DataFrame): The training DataFrame.
        test (pl.DataFrame): The testing DataFrame.

    Returns:
        pl.DataFrame: A single DataFrame with vertically stacked data from common columns.
    """
    common_columns: list[str] = [col for col in train.columns if col in test.columns]
    
    return pl.concat([train.select(common_columns), test.select(common_columns)], how="vertical")

def split_dataset(train: pl.DataFrame, test: pl.DataFrame, features: list[str]) -> DatasetOutput: 
    """
    Splits the data into features (X) and target (y), and scales the features.

    Args:
        train (pl.DataFrame): The processed training DataFrame.
        test (pl.DataFrame): The processed testing DataFrame.
        features (list[str]): List of features to used in model. 

    Returns:
        DatasetOutput: A dataclass containing the scaled feature sets, target series, and the fitted scaler.
    """
    X_train = train.drop(['date_id','target']) 
    y_train = train.get_column('target')
    X_test = test.drop(['date_id','target']) 
    y_test = test.get_column('target')
    
    scaler = StandardScaler() 
    
    X_train_scaled_np = scaler.fit_transform(X_train)
    X_train = pl.from_numpy(X_train_scaled_np, schema=features)
    
    X_test_scaled_np = scaler.transform(X_test)
    X_test = pl.from_numpy(X_test_scaled_np, schema=features)
    
    
    return DatasetOutput(
        X_train = X_train,
        y_train = y_train, 
        X_test = X_test, 
        y_test = y_test,
        scaler = scaler
    )

In [20]:
def convert_ret_to_signal(
    ret_arr: np.ndarray,
    params: RetToSignalParameters
) -> np.ndarray:
    """
    Converts raw model predictions (expected returns) into a trading signal.

    Args:
        ret_arr (np.ndarray): The array of predicted returns.
        params (RetToSignalParameters): Parameters for scaling and clipping the signal.

    Returns:
        np.ndarray: The resulting trading signal, clipped between min and max values.
    """
    return np.clip(
        ret_arr * params.signal_multiplier + 1, params.min_signal, params.max_signal
    )

In [21]:
train: pl.DataFrame = load_trainset()
test: pl.DataFrame = load_testset() 
print(train.tail(3)) 
print(test.head(3))

shape: (3, 98)
┌─────────┬─────┬─────┬─────┬───┬───────────┬─────────────────┬────────────────┬───────────┐
│ date_id ┆ D1  ┆ D2  ┆ D3  ┆ … ┆ V9        ┆ forward_returns ┆ risk_free_rate ┆ target    │
│ ---     ┆ --- ┆ --- ┆ --- ┆   ┆ ---       ┆ ---             ┆ ---            ┆ ---       │
│ i64     ┆ f64 ┆ f64 ┆ f64 ┆   ┆ f64       ┆ f64             ┆ f64            ┆ f64       │
╞═════════╪═════╪═════╪═════╪═══╪═══════════╪═════════════════╪════════════════╪═══════════╡
│ 9008    ┆ 0.0 ┆ 0.0 ┆ 0.0 ┆ … ┆ -0.530228 ┆ -0.002897       ┆ 0.0001525      ┆ -0.003362 │
│ 9009    ┆ 0.0 ┆ 0.0 ┆ 0.0 ┆ … ┆ -0.512769 ┆ -0.027028       ┆ 0.000153       ┆ -0.027493 │
│ 9010    ┆ 0.0 ┆ 0.0 ┆ 0.0 ┆ … ┆ -0.015503 ┆ 0.015344        ┆ 0.000153       ┆ 0.014879  │
└─────────┴─────┴─────┴─────┴───┴───────────┴─────────────────┴────────────────┴───────────┘
shape: (3, 99)
┌─────────┬─────┬─────┬─────┬───┬───────────┬───────────┬─────────────────────┬────────────────────┐
│ date_id ┆ D1  ┆ D2  ┆ D3  ┆ … 

In [22]:
df: pl.DataFrame = join_train_test_dataframes(train, test)
df = create_example_dataset(df=df) 
train: pl.DataFrame = df.filter(pl.col('date_id').is_in(train.get_column('date_id')))
test: pl.DataFrame = df.filter(pl.col('date_id').is_in(test.get_column('date_id')))

FEATURES: list[str] = [col for col in test.columns if col not in ['date_id', 'target']]

dataset: DatasetOutput = split_dataset(train=train, test=test, features=FEATURES) 

X_train: pl.DataFrame = dataset.X_train
X_test: pl.DataFrame = dataset.X_test
y_train: pl.DataFrame = dataset.y_train
y_test: pl.DataFrame = dataset.y_test
scaler: StandardScaler = dataset.scaler 

C:\Users\53460\AppData\Local\Temp\ipykernel_43760\102800950.py:3: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  train: pl.DataFrame = df.filter(pl.col('date_id').is_in(train.get_column('date_id')))
C:\Users\53460\AppData\Local\Temp\ipykernel_43760\102800950.py:4: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  test: pl.DataFrame = df.filter(pl.col('date_id').is_in(test.get_column('date_id')))


In [23]:
# -----------------------------
# Model: Ridge Regression (L2)
# -----------------------------
from typing import Any

def _to_numpy(x: Any) -> np.ndarray:
    """Robust conversion helper for polars/pandas/numpy inputs."""
    if hasattr(x, "to_numpy"):
        return x.to_numpy()
    return np.asarray(x)

X_train_np = _to_numpy(X_train)
y_train_np = _to_numpy(y_train)

model_cv: RidgeCV = RidgeCV(
    alphas=ridge_params.alphas,
    cv=ridge_params.cv
)
model_cv.fit(X_train_np, y_train_np)

best_alpha: float = float(model_cv.alpha_)
print(f"[RidgeCV] best alpha = {best_alpha:.6g}")

# Fit the final Ridge model using the best alpha found by cross-validation
model: Ridge = Ridge(alpha=best_alpha)
model.fit(X_train_np, y_train_np)


[RidgeCV] best alpha = 100


Ridge(alpha=100.0)

In [24]:
def predict(test: pl.DataFrame) -> float:
    test = test.rename({'lagged_forward_returns':'target'})
    df: pl.DataFrame = create_example_dataset(test)
    X_test: pl.DataFrame = df.select(FEATURES)

    X_test_np: np.ndarray = X_test.to_numpy()
    X_test_scaled_np: np.ndarray = scaler.transform(X_test_np)

    raw_pred: float = float(model.predict(X_test_scaled_np)[0])
    return convert_ret_to_signal(raw_pred, ret_signal_params)


In [25]:
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway((str(DATA_PATH),))

c:\Users\53460\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\53460\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\53460\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\53460\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\53460\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\53460\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid f

In [26]:
#打印预测结果
pd.read_parquet('./submission.parquet').head()

,date_id,prediction
0,8980,0.890947
1,8981,1.021231
2,8982,1.017714
3,8983,1.004054
4,8984,1.030538


## 生成Submission并计算Sharpe Ratio得分

In [27]:
# 生成完整的submission DataFrame
import pandas as pd

# 获取测试集的所有预测
test_original = pl.read_csv(DATA_PATH / "test.csv")
test_predictions = []

for row in test_original.iter_rows(named=True):
    # 为每一行创建一个DataFrame
    single_row_df = pl.DataFrame([row])
    # 使用predict函数生成预测
    pred = predict(single_row_df)
    test_predictions.append(pred)

# 创建submission DataFrame
submission_df = pd.DataFrame({
    'date_id': test_original['date_id'].to_list(),
    'row_id': test_original['date_id'].to_list(),
    'prediction': test_predictions
})

print("Submission DataFrame:")
print(submission_df.head(10))
print(f"\nSubmission shape: {submission_df.shape}")
print(f"\nPrediction statistics:")
print(submission_df['prediction'].describe())

Submission DataFrame:
   date_id  row_id  prediction
0     8980    8980    0.890947
1     8981    8981    1.021231
2     8982    8982    1.017714
3     8983    8983    1.004054
4     8984    8984    1.030538
5     8985    8985    1.040557
6     8986    8986    0.965255
7     8987    8987    1.126943
8     8988    8988    0.918714
9     8989    8989    0.958898

Submission shape: (10, 3)

Prediction statistics:
count    10.000000
mean      0.997485
std       0.067335
min       0.890947
25%       0.960487
50%       1.010884
75%       1.028211
max       1.126943
Name: prediction, dtype: float64


c:\Users\53460\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\53460\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\53460\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\53460\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\53460\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\53460\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid f

In [28]:
# 创建solution DataFrame（包含实际的forward_returns和risk_free_rate）
solution_df = test_original.select([
    'date_id', 
    pl.col('lagged_forward_returns').alias('forward_returns'),
    pl.col('lagged_risk_free_rate').alias('risk_free_rate')
]).to_pandas()

solution_df['row_id'] = solution_df['date_id']

print("Solution DataFrame:")
print(solution_df.head())
print(f"\nForward returns 统计:")
print(solution_df['forward_returns'].describe())

Solution DataFrame:
   date_id  forward_returns  risk_free_rate  row_id
0     8980         0.003541        0.000161    8980
1     8981        -0.005964        0.000162    8981
2     8982        -0.007410        0.000160    8982
3     8983         0.005420        0.000160    8983
4     8984         0.008357        0.000159    8984

Forward returns 统计:
count    10.000000
mean      0.001702
std       0.005482
min      -0.007410
25%      -0.001594
50%       0.002674
75%       0.004950
max       0.008357
Name: forward_returns, dtype: float64


In [29]:
# 导入SharpeRatio评分函数并计算得分
from SharpeRatio import score

try:
    sharpe_score = score(
        solution=solution_df, 
        submission=submission_df, 
        row_id_column_name='date_id'
    )
    print(f"{'='*60}")
    print(f"✓ Sharpe Ratio Score: {sharpe_score:.6f}")
    print(f"{'='*60}")
    
    # 显示策略详情
    print(f"\n策略表现:")
    print(f"  - 平均预测信号: {submission_df['prediction'].mean():.4f}")
    print(f"  - 信号标准差: {submission_df['prediction'].std():.4f}")
    print(f"  - 信号范围: [{submission_df['prediction'].min():.4f}, {submission_df['prediction'].max():.4f}]")
    
except Exception as e:
    print(f"❌ 计算Sharpe Ratio时出错: {e}")
    import traceback
    traceback.print_exc()

✓ Sharpe Ratio Score: 3.975119

策略表现:
  - 平均预测信号: 0.9975
  - 信号标准差: 0.0673
  - 信号范围: [0.8909, 1.1269]
